# 🧠 LasmoidV1 — Production Model Pre-Training on Kaggle T4 GPU x2

**Developer:** theory903 (Abhishek Jha)  
**Architecture:** MLA + mHC + MoE + CQRS + MTP (Neuro-Symbolic MoE)  
**Tokenizer:** GPT-2 (vocab size: 50,257)  
**Context Window:** up to 1024 tokens  
**Weights & Optimizers:** Nesterov Muon for 2D weight matrices & AdamW for embed/bias parameters  
**SOTA Features:** EMA weight averaging, gradient checkpointing, W&B tracking, 8-stream curriculum, WSD schedule  
**Checkpoints:** Auto-synced to Hugging Face Hub (saves your progress across Kaggle's 12-hour session timeout limits)

### Supported Configurations
- **100M size**: `dim=512`, `n_layers=8`, `n_heads=8` — ~100.7M parameters
- **300M size**: `dim=768`, `n_layers=13`, `n_heads=12` — ~303.0M parameters (classic GPT-medium width with depth-first MoE)

### Prerequisites
Add `HF_TOKEN` as a Kaggle Secret (Settings → Secrets → Add New Secret).  
Name it exactly: **`HF_TOKEN**  
For W&B tracking: `WANDB_API_KEY` as a Kaggle Secret (optional)

In [ ]:
# ── CELL 1: Environment diagnostics ───────────────────────────────────
import subprocess, sys, os

print('=== GPU Info ===')
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU found!')

print('=== Python ===')
print(sys.version)

print('=== Disk Space ===')
subprocess.run(['df', '-h', '/kaggle/working'], text=True)

print('=== RAM ===')
with open('/proc/meminfo') as f:
    for line in f:
        if 'MemTotal' in line or 'MemAvailable' in line:
            print(line.strip())

In [ ]:
# ── CELL 2: Install dependencies ───────────────────────────────────────
# Install core deps + optional W&B for experiment tracking
!pip install -q tiktoken datasets transformers huggingface_hub accelerate wandb
print('All dependencies installed ✓')

In [ ]:
# ── CELL 3: Clone/Pull repo from GitHub ─────────────────────────────────────
import os

REPO_URL = 'https://github.com/Theory903/Lasmoid-V1.git'
WORK_DIR = '/kaggle/working/Lasmoid-V1'

if not os.path.exists(WORK_DIR):
    result = os.system(f'git clone {REPO_URL} {WORK_DIR}')
    if result != 0:
        print('Git clone failed. You may need to upload files manually (see Cell 3b).')
    else:
        print(f'Cloned repo to {WORK_DIR} ✓')
else:
    print(f'Repo already exists at {WORK_DIR}')
    os.system(f'cd {WORK_DIR} && git pull origin main')

In [ ]:
# ── CELL 3b: ALTERNATIVE — Upload files manually ───────────────────────
# If you don't have a public GitHub repo, use Kaggle's "Add Input" → "Upload"
# to add a ZIP of your project, then unpack it.
#
# import shutil
# shutil.unpack_archive('/kaggle/input/lasmoid-v1/Lasmoid-V1.zip', '/kaggle/working/Lasmoid-V1')
print('If you used git clone in Cell 3, skip this cell.')

In [ ]:
# ── CELL 4: Hugging Face + W&B Authentication via Kaggle Secrets ───────
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi

secrets = UserSecretsClient()

# ── HF Hub (required) ────────────────────────────────────────────────
HF_TOKEN = secrets.get_secret('HF_TOKEN')
login(token=HF_TOKEN, add_to_git_credential=False)
api = HfApi(token=HF_TOKEN)
username = api.whoami()['name']
os.environ['HF_TOKEN'] = HF_TOKEN
print(f'Logged in as: {username} ✓')
print(f'Checkpoints will sync to: {username}/lasmoid-<model_size>')

# ── W&B (optional) ───────────────────────────────────────────────────
try:
    WANDB_KEY = secrets.get_secret('WANDB_API_KEY')
    os.environ['WANDB_API_KEY'] = WANDB_KEY
    print('W&B: authenticated ✓')
except Exception:
    print('W&B: no WANDB_API_KEY secret found (tracking disabled)')

In [ ]:
# ── CELL 5: Verify model initialization ──────────────────────
import sys
WORK_DIR = '/kaggle/working/Lasmoid-V1'
sys.path.insert(0, WORK_DIR)

import torch
from train_kaggle import MODEL_CONFIGS
from inference.model import LasmoidV1, ModelArgs

# Select model size: "100M" or "300M"
MODEL_SIZE = "300M"

cfg = MODEL_CONFIGS[MODEL_SIZE].copy()
cfg.pop("_verified_params", None)
cfg["max_seq_len"] = 1024
cfg["max_batch_size"] = 4

model_args = ModelArgs(**cfg)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = LasmoidV1(model_args).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Device: {device}')
print(f'Model: {MODEL_SIZE} | LasmoidV1, {total_params:,} params')
print(f'Developer: theory903 (Abhishek Jha)')

# Quick forward pass sanity check
x = torch.randint(0, 50257, (2, 64)).to(device)
with torch.no_grad():
    out = model(x, x)
print(f'Forward pass output shape: {out[0].shape}')
print('Model initialized successfully ✓')

# ── Gradient checkpointing test (optional memory saver) ───────────
if hasattr(model, 'gradient_checkpointing_enable'):
    model.gradient_checkpointing_enable()
    with torch.no_grad():
        out2 = model(x, x)
    print(f'Gradient checkpointing: OK, output shape {out2[0].shape}')

In [ ]:
# ── CELL 6: LAUNCH TRAINING ────────────────────────
#
# LasmoidV1 300M — Production Training
# by theory903 (Abhishek Jha)
# ─────────────────────────────
# Model  : dim=768, L=13, H=12 → 303.0M parameters (verified)
# Budget : 200K steps × 4 seqs × 8 accum × 1024 tok ≈ 6.5B tokens
#          (≈ 3.25× Chinchilla-optimal for 300M; very solid run!)
#
# Curriculum stages (auto-managed by script):
#   8 streams: UltraFineWeb-L3, UltraData-IF, UltraData-Math, UltraData-Code,
#              claude_mythos, DeepThink, smollm python-edu, FineWeb-Edu
#
#   Stage 1: 0%–70%   broad pretraining — web + edu + raw code + light SFT
#   Stage 2: 70%–90%  quality annealing — SFT + reasoning + mythos
#   Stage 3: 90%–100% final cooldown   — pure quality SFT + reasoning
#
# Schedule: WSD (Warmup-Stable-Decay) — stable plateau prevents regression on restart
#
# SOTA features: gradient checkpointing | EMA weight avg | W&B tracking
# HF Hub sync every 500 steps; session watchdog exits at 11.5h for auto-resume
#
# TIP: Run `python train_kaggle.py --data_check --model_size 300M` first
#      to verify all 8 dataset streams open correctly before full training.
#
import subprocess, os, sys

WORK_DIR = '/kaggle/working/Lasmoid-V1'
os.makedirs(f'{WORK_DIR}/checkpoints', exist_ok=True)

# Pull latest code from GitHub
os.system(f'cd {WORK_DIR} && git pull origin main')

cmd = [
    'torchrun', '--nproc_per_node=2', f'{WORK_DIR}/train_kaggle.py',
    '--model_size', '300M',
    '--max_iters',     '200000',
    '--batch_size',    '4',
    '--grad_accum',    '8',
    '--lr',            '3e-4',
    '--muon_lr',       '2e-3',
    '--warmup_steps',  '2000',
    '--decay_frac',    '0.2',
    '--lr_min_ratio',  '0.1',
    '--weight_decay',  '0.1',
    '--grad_clip',     '1.0',
    '--dtype',         'bf16',
    '--gradient_checkpointing',
    '--ema_decay',     '0.9999',
    '--wandb',         'lasmoid-v1',
    '--save_interval', '500',
    '--log_interval',  '10',
    '--session_hours', '11.5',
    '--checkpoint_dir', f'{WORK_DIR}/checkpoints',
]

print('Starting LasmoidV1 300M training...')
print(f'Command: {" ".join(cmd)}')
print('=' * 70)

process = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, cwd=WORK_DIR)

for line in process.stdout:
    print(line, end='', flush=True)

process.wait()
print(f'\nTraining finished with return code: {process.returncode}')

In [ ]:
# ── CELL 7: Verify checkpoint on HF Hub ──────────────────────
# Run this anytime to see what's saved remotely
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret('HF_TOKEN')
api = HfApi()
username = api.whoami(token=HF_TOKEN)['name']

# Change this to match the model size you trained: "100M" or "300M"
MODEL_SIZE = "300M"
repo_id = f'{username}/lasmoid-{MODEL_SIZE.lower()}'

try:
    files = list(api.list_repo_files(repo_id=repo_id, token=HF_TOKEN))
    pt_files = [f for f in files if f.endswith('.pt')]
    csv_files = [f for f in files if f.endswith('.csv')]
    print(f'Repository: {repo_id}')
    print(f'Checkpoints found ({len(pt_files)}):')
    for f in sorted(pt_files):
        print(f'  - {f}')
    if csv_files:
        print(f'\nMetrics CSV: {csv_files[0]} (download for loss curve plot)')
    if not pt_files:
        print('No checkpoints yet — training may still be in progress.')
except Exception as e:
    print(f'Error: {e}')

In [ ]:
# ── CELL 8: Quick generation test after training ─────────────
# Note: if --ema_decay was set, final checkpoint may contain EMA-smoothed weights
import sys, torch, tiktoken, glob
WORK_DIR = '/kaggle/working/Lasmoid-V1'
sys.path.insert(0, WORK_DIR)

from train_kaggle import MODEL_CONFIGS
from inference.model import LasmoidV1, ModelArgs

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = tiktoken.get_encoding('gpt2')

# Select model size to load: "100M" or "300M"
MODEL_SIZE = "300M"

cfg = MODEL_CONFIGS[MODEL_SIZE].copy()
cfg.pop("_verified_params", None)
cfg["max_seq_len"] = 1024
cfg["max_batch_size"] = 1

model_args = ModelArgs(**cfg)

# Load latest checkpoint
ckpts = sorted(glob.glob(f'{WORK_DIR}/checkpoints/lasmoid_checkpoint_step_*.pt'))
if not ckpts:
    ckpts = glob.glob(f'{WORK_DIR}/checkpoints/lasmoid_latest.pt')

if ckpts:
    latest = ckpts[-1]
    print(f'Loading: {latest}')
    model = LasmoidV1(model_args).to(device)
    ckpt = torch.load(latest, map_location=device)
    state_dict = ckpt.get('model_state_dict', ckpt)
    
    # Strip compilation prefix if any
    clean_state_dict = {}
    for k, v in state_dict.items():
        if k.startswith('_orig_mod.'):
            clean_state_dict[k[10:]] = v
        else:
            clean_state_dict[k] = v
            
    model.load_state_dict(clean_state_dict)
    model.eval()

    prompt = 'The meaning of intelligence is'
    tokens = tokenizer.encode(prompt)
    x = torch.tensor([tokens], dtype=torch.long).to(device)

    print(f'Prompt: {prompt}')
    print('Generation:', end=' ')
    with torch.no_grad():
        for _ in range(50):
            logits, _, _, _ = model(x, x)
            next_tok = logits[0, -1].argmax().item()
            print(tokenizer.decode([next_tok]), end='', flush=True)
            x = torch.cat([x, torch.tensor([[next_tok]]).to(device)], dim=1)
            if x.shape[1] >= model_args.max_seq_len:
                break
    print()
else:
    print('No checkpoint found. Run Cell 6 first.')

In [ ]:
# ── CELL 9: Quick smoke test (50 steps) ───────────────────────
# Run this BEFORE Cell 6 to verify everything works end-to-end.
# Trains for 50 steps on real data, saves checkpoint, tests inference.
# by theory903 (Abhishek Jha) — Lasmoid-V1
import subprocess, os, sys, glob, glob

WORK_DIR = '/kaggle/working/Lasmoid-V1'

# Step 1: Verify all 8 data streams open correctly
print('=== Step 1: --data_check (verify 8 streams) ===')
subprocess.run([
    'torchrun', '--nproc_per_node=2', f'{WORK_DIR}/train_kaggle.py',
    '--model_size', '300M',
    '--data_check',
], cwd=WORK_DIR, check=True)
print('Data check passed!')

# Step 2: Train 50 steps with full pipeline (incl. EMA + grad ckpt + bf16)
print('\n=== Step 2: 50-step training smoke test ===')
cmd = [
    sys.executable, f'{WORK_DIR}/train_kaggle.py',
    '--model_size', '300M',
    '--max_iters',      '50',
    '--batch_size',     '2',
    '--grad_accum',     '2',
    '--lr',             '3e-4',
    '--muon_lr',        '2e-3',
    '--warmup_steps',   '10',
    '--decay_frac',     '0.2',
    '--lr_min_ratio',   '0.1',
    '--weight_decay',   '0.1',
    '--grad_clip',      '1.0',
    '--dtype',          'bf16',
    '--gradient_checkpointing',
    '--ema_decay',      '0.9999',
    '--save_interval',  '25',
    '--log_interval',   '5',
    '--checkpoint_dir', f'{WORK_DIR}/checkpoints_test',
]
proc = subprocess.run(cmd, cwd=WORK_DIR)
assert proc.returncode == 0, f'Training failed: {proc.returncode}'
print('Training smoke test passed!')

# Step 3: Verify artifacts exist
print('\n=== Step 3: Verify artifacts ===')
ckpts = sorted(glob.glob(f'{WORK_DIR}/checkpoints_test/*.pt'))
csvs = sorted(glob.glob(f'{WORK_DIR}/checkpoints_test/*.csv'))
print(f'Checkpoints: {len(ckpts)}')
print(f'Metrics CSVs: {len(csvs)}')
sizes = [os.path.getsize(f) / 1e6 for f in ckpts]
print(f'Checkpoint sizes (MB): {[round(s, 1) for s in sizes]}')

assert len(ckpts) >= 1, 'No checkpoint saved'
assert len(csvs) >= 1, 'No metrics CSV saved'
assert sizes[-1] > 100, f'Checkpoint too small: {sizes[-1]:.1f}MB'
print('All artifacts verified!')

print('\n=== SMOKE TEST PASSED ✅ ===')
print('If all checks passed, proceed to Cell 6 for full training.')
